# Fase 7b — Las reglas *woman-child-group*: rompiendo el 0.80

La pregunta que motivó esta iteración: **"¿no podríamos mejorarlo?"**. Respuesta: sí — refinando la
mejor idea que teníamos (los grupos compartían destino) con una observación histórica más fina.

## La idea: no todo el grupo compartía destino — las mujeres y niños sí

`FamilySurvival` (Fase 7) miraba al grupo entero. Pero piensa en la mecánica del naufragio: quienes
esperaban juntos, subían juntos al bote o se quedaban juntos eran las **mujeres y los niños** del
grupo — "women and children first" operaba sobre ellos como *unidad*. Los hombres adultos del grupo
tenían su propia suerte (mayormente mala), y meterlos en el cálculo añade ruido.

De ahí las reglas **WCG** (*woman-child groups*), un clásico de la comunidad de Kaggle:

```
Por defecto:  regla del género (mujer vive, hombre muere)
Excepción 1:  niño (Master) cuyo grupo mujer-niño conocido sobrevivió ENTERO -> vive
Excepción 2:  mujer cuyo grupo mujer-niño conocido murió ENTERO             -> muere
```

Fíjate en lo radical del planteamiento: es un "modelo" de **cero parámetros aprendidos**. Solo
género + 2 excepciones quirúrgicas donde tenemos la información más fiable que existe en este
problema. Todo implementado en [`src/grupos.py`](../src/grupos.py) (`es_wc`, `wc_flags`,
`regla_wcg`), con las mismas dos disciplinas anti-leakage de la Fase 7: exclusión de uno mismo y
cálculo fold-aware en CV.

## Los cuatro candidatos

Con 3 submissions restantes, preparamos 4 candidatos y elegimos por CV honesta:

1. **Regla WCG pura** — la descrita arriba.
2. **SVM + FS + flags WC como features** — dejar que el modelo pondere las nuevas señales.
3. **Híbrido** — predicción del SVM+FS (nuestro campeón, LB 0.79425) con las reglas WCG como
   override quirúrgico encima.
4. SVM + FS (referencia).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import sys
sys.path.append("../src")
import validacion, grupos

from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

train_l = pd.read_csv("../data/processed/train_limpio.csv")
test_l  = pd.read_csv("../data/processed/test_limpio.csv")
train_f = pd.read_csv("../data/processed/train_features.csv")
test_f  = pd.read_csv("../data/processed/test_features.csv")
X = train_f.drop(columns=["PassengerId", "Survived"])
y = train_f["Survived"]

svm_base = make_pipeline(StandardScaler(), SVC())

res = {"Regla WCG pura": [], "SVM+FS+WC (features)": [], "Hibrido (SVM+FS -> WCG)": [], "SVM+FS (referencia)": []}
for idx_tr, idx_va in validacion.CV.split(X, y):
    conocidos = train_l.iloc[idx_tr]                       # fold-aware, como siempre
    fs = grupos.family_survival(conocidos, train_l)
    flags = grupos.wc_flags(conocidos, train_l)

    X2 = X.assign(FamilySurvival=fs.loc[train_f.PassengerId].values)
    X3 = X2.assign(WCAllLived=flags.loc[train_f.PassengerId, "WCAllLived"].values,
                   WCAllDied=flags.loc[train_f.PassengerId, "WCAllDied"].values)
    va_f, y_va = train_f.iloc[idx_va], y.iloc[idx_va].values

    res["Regla WCG pura"].append((grupos.regla_wcg(va_f, flags) == y_va).mean())
    pred_svm = clone(svm_base).fit(X2.iloc[idx_tr], y.iloc[idx_tr]).predict(X2.iloc[idx_va])
    res["SVM+FS (referencia)"].append((pred_svm == y_va).mean())
    res["SVM+FS+WC (features)"].append(
        (clone(svm_base).fit(X3.iloc[idx_tr], y.iloc[idx_tr]).predict(X3.iloc[idx_va]) == y_va).mean())

    pred_h = pred_svm.copy()                                # hibrido: override quirurgico
    f_va = flags.loc[va_f["PassengerId"]].reset_index(drop=True)
    pred_h[(va_f["Title_Master"].values == 1) & (f_va["WCAllLived"].values == 1)] = 1
    pred_h[(va_f["IsFemale"].values == 1) & (f_va["WCAllDied"].values == 1)] = 0
    res["Hibrido (SVM+FS -> WCG)"].append((pred_h == y_va).mean())

print("CV honesta (fold-aware):")
for n, s in res.items():
    s = np.array(s)
    print(f"  {n:26} {s.mean():.4f} +- {s.std():.4f}")

CV honesta (fold-aware):
  Regla WCG pura             0.8429 +- 0.0182
  SVM+FS+WC (features)       0.8507 +- 0.0156
  Hibrido (SVM+FS -> WCG)    0.8496 +- 0.0177
  SVM+FS (referencia)        0.8473 +- 0.0154


La regla pura da **0.8429 de CV — mejor que la regresión logística con 16 features (0.8328)** y
a milímetros de modelos con todo nuestro arsenal. Con dos excepciones y cero aprendizaje.

Un apunte importante antes de ver el LB: para las reglas de grupo, **la CV es pesimista por
construcción**. En cada fold solo "conocemos" al 80% de los pasajeros — un fold ve menos compañeros
de grupo que el caso real del test, donde el train entero es conocido. Es la situación inversa a la
del sobreajuste: aquí el examen real juega ligeramente a nuestro favor.

## El veredicto del leaderboard

Subidas las 3 el 2026-07-08 (cuota del día agotada: 10/10):

| Versión | CV honesta | **LB público** |
|---|---|---|
| **v12 Regla WCG pura** | 0.8429 ± 0.018 | **0.80143** 🏆 |
| v11 Híbrido (SVM+FS → WCG) | 0.8496 ± 0.018 | 0.79425 |
| v10 SVM + FS + flags WC | 0.8507 ± 0.016 | 0.79186 |
| (v06 SVM + FS, campeón anterior) | 0.8473 ± 0.015 | 0.79425 |

**Ganó la regla pura — el candidato con PEOR CV de los tres.** Y no es casualidad; es la culminación
de todo lo que este proyecto lleva enseñando:

1. **Menos capacidad = menos que perder en el salto CV→LB.** La regla no tiene parámetros que
   sobreajustar: su caída CV→LB (−4.2 pts) fue la menor de todos nuestros modelos ML con features
   de grupo (−5.3 a −6.4). El orden por CV se invirtió otra vez al cruzar al mundo real — y otra
   vez a favor del más simple.
2. **El SVM diluye la señal; la regla la aplica pura.** Para el SVM, `WCAllDied` es una feature más
   que ponderar entre 19; la regla la trata como lo que es: información casi determinista.
3. **El conocimiento del dominio vale oro.** La mejora no vino de más matemáticas sino de entender
   *cómo* se murió en el Titanic: por grupos, con las mujeres y niños como unidad.

## El viaje final del proyecto

| Hito | LB |
|---|---|
| Regla del género | 0.76555 |
| Mejor ML con 16 features (Fases 2-6) | 0.76794 |
| ML + FamilySurvival (Fase 7) | 0.79425 |
| **Regla WCG pura (Fase 7b)** | **0.80143** |

**0.80143** nos sitúa cómodamente en la zona alta del leaderboard honesto (el rango legítimo muere
~0.83; lo que hay por encima ya sabes lo que es). La ironía final y la mejor lección del proyecto:
tras ocho fases de pipelines, torneos y tuning, el podio es para *género + 2 excepciones bien
elegidas*. Los modelos complejos no eran el objetivo — eran el camino para saber **encontrar,
validar y confiar** en algo así de simple.

## Registro final

In [2]:
registro = pd.read_csv("../submissions/registro_experimentos.csv")
nuevos = pd.DataFrame([
    {"version": "v10-svm-wc",    "fecha": "2026-07-08", "features": "16 + FS + flags WC", "modelo": "StandardScaler+SVC",              "cv_media": 0.8507, "cv_std": 0.0156, "lb_publico": 0.79186},
    {"version": "v11-hibrido",   "fecha": "2026-07-08", "features": "16 + FS",            "modelo": "SVM+FS con override WCG",          "cv_media": 0.8496, "cv_std": 0.0177, "lb_publico": 0.79425},
    {"version": "v12-regla-wcg", "fecha": "2026-07-08", "features": "genero + grupos WC", "modelo": "Regla pura (sin ML) - GANADORA",   "cv_media": 0.8429, "cv_std": 0.0182, "lb_publico": 0.80143},
])
registro = registro[~registro["version"].isin(nuevos["version"])]
registro = pd.concat([registro, nuevos], ignore_index=True)
registro.to_csv("../submissions/registro_experimentos.csv", index=False)
registro.sort_values("lb_publico", ascending=False, na_position="last")

,version,fecha,features,modelo,cv_media,cv_std,lb_publico
13,v12-regla-wcg,2026-07-08,genero + grupos WC,Regla pura (sin ML) - GANADORA,0.8429,0.0182,0.80143
7,v06-svm-fs,2026-07-08,16 + FamilySurvival,StandardScaler+SVC,0.8473,0.0154,0.79425
12,v11-hibrido,2026-07-08,16 + FS,SVM+FS con override WCG,0.8496,0.0177,0.79425
11,v10-svm-wc,2026-07-08,16 + FS + flags WC,StandardScaler+SVC,0.8507,0.0156,0.79186
8,v07-rf-fs,2026-07-08,16 + FamilySurvival,"RF(depth=8, leaf=4, feat=0.5)",0.8507,0.0137,0.78468
9,v08-knn-fs,2026-07-08,16 + FamilySurvival,KNN(k=9) escalado,0.8451,0.0092,0.77990
10,v09-gb-fs,2026-07-08,16 + FamilySurvival,"GB(lr=0.03, depth=4, n=100)",0.8563,0.0159,0.77511
3,v02-rf-afinado,2026-07-08,16 (Fase 3),"RF(depth=8, leaf=4, feat=0.5)",0.8473,0.0092,0.76794
6,v05-logistica,2026-07-08,16 (Fase 3),StandardScaler+LogisticRegression,0.8328,0.0110,0.76794
1,base-genero,2026-07-08,IsFemale,ReglaGenero,0.7868,0.0188,0.76550
